DeNovoSeer

#Xiyu Rao
#2026-04-20

In [ ]:
"""
DeNovoSeer main training script with final contrastive augmentation (noise + masking)
and supplementary prediction export.

"""

%matplotlib inline

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    auc,
)

# =========================
# Global hyperparameters
# =========================
EPOCHS = 50
BATCH_SIZE = 256
LABELED_RATIO = 0.125
LABELED_BATCH_SIZE = int(BATCH_SIZE * LABELED_RATIO)
UNLABELED_BATCH_SIZE = BATCH_SIZE - LABELED_BATCH_SIZE

LEARNING_RATE = 5e-4
LAMBDA_RECON = 1.0
LAMBDA_CONTRAST = 0.5
PATIENCE = 5
MIN_DELTA = 0.001

# Final contrastive augmentation selected by ablation: noise + masking
CONTRAST_NOISE_STD = 0.1
CONTRAST_MASK_PROB = 0.05
USE_CONTRAST_NOISE = True
USE_CONTRAST_MASKING = True

PLOT_CURVES = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cycle_loader(loader):
    """Infinite batch iterator."""
    while True:
        for batch in loader:
            yield batch


def apply_contrastive_augmentation(
    x,
    noise_std=0.1,
    mask_prob=0.05,
    use_noise=True,
    use_masking=True,
):
    """Feature-space contrastive augmentation: Gaussian noise + random masking."""
    x_aug = x.clone()
    if use_noise:
        x_aug = x_aug + torch.randn_like(x_aug) * noise_std
    if use_masking:
        mask = (torch.rand_like(x_aug) > mask_prob).float()
        x_aug = x_aug * mask
    return x_aug


# =========================
# Load processed data
# =========================
# replace with your actual DATA_path
DATA_PATH = "processed_coding_data_scaled.csv"
df = pd.read_csv(DATA_PATH)

non_numeric_columns = [
    "ClinVar_label",
    "variant_index",
    "Otherinfo",
    "Phenotype",
    "platform",
    "study",
    "PMID",
    "Explanation",
]
available_non_numeric = [col for col in non_numeric_columns if col in df.columns]

numeric_feature_names = [col for col in df.columns if col not in available_non_numeric and col != "label"]
labels_all = df["label"].values
features_all = df[numeric_feature_names].values
non_numeric_all = df[available_non_numeric].copy() if available_non_numeric else pd.DataFrame(index=df.index)

print("Loaded data from:", DATA_PATH)
print("Total samples:", len(df))
print("Numeric feature count:", len(numeric_feature_names))
print("Non-numeric column count:", len(available_non_numeric))
print("Feature example count:", len(numeric_feature_names))


# =========================
# Dataset and DataLoader
# =========================
class MutDataset(Dataset):
    def __init__(self, features, labels):
        self.features = np.asarray(features, dtype=np.float32)
        self.labels = np.asarray(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        sample_idx = torch.tensor(idx, dtype=torch.long)
        return x, y, sample_idx


def build_train_loaders(train_pack, labeled_batch_size, unlabeled_batch_size, seed):
    labeled_dataset = MutDataset(train_pack["X_labeled"], train_pack["y_labeled"])
    unlabeled_dataset = MutDataset(train_pack["X_unlabeled"], train_pack["y_unlabeled"])

    g1 = torch.Generator()
    g1.manual_seed(seed)
    g2 = torch.Generator()
    g2.manual_seed(seed + 123)

    labeled_loader = DataLoader(
        labeled_dataset,
        batch_size=labeled_batch_size,
        shuffle=True,
        drop_last=True,
        generator=g1,
    )
    unlabeled_loader = DataLoader(
        unlabeled_dataset,
        batch_size=unlabeled_batch_size,
        shuffle=True,
        drop_last=True,
        generator=g2,
    )
    return labeled_loader, unlabeled_loader


def build_eval_loader(features, labels, batch_size=64):
    dataset = MutDataset(features, labels)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return loader


def oversample_labeled_subset(train_feats, train_labels, train_non_numeric, seed=42):
    labeled_mask = train_labels != -1
    unlabeled_mask = train_labels == -1

    X_labeled = train_feats[labeled_mask]
    y_labeled = train_labels[labeled_mask]
    nn_labeled = train_non_numeric.iloc[np.where(labeled_mask)[0]].reset_index(drop=True)

    X_unlabeled = train_feats[unlabeled_mask]
    y_unlabeled = train_labels[unlabeled_mask]
    nn_unlabeled = train_non_numeric.iloc[np.where(unlabeled_mask)[0]].reset_index(drop=True)

    ros = RandomOverSampler(random_state=seed)
    X_labeled_res, y_labeled_res = ros.fit_resample(X_labeled, y_labeled)
    nn_labeled_res = nn_labeled.iloc[ros.sample_indices_].reset_index(drop=True)

    return {
        "X_labeled": X_labeled_res,
        "y_labeled": y_labeled_res,
        "nn_labeled": nn_labeled_res,
        "X_unlabeled": X_unlabeled,
        "y_unlabeled": y_unlabeled,
        "nn_unlabeled": nn_unlabeled,
    }


# =========================
# Model
# =========================
class SemiSupervisedHybridCNNIDCNN(nn.Module):
    def __init__(self, input_channels=1, num_classes=2, feature_dim=305):
        super().__init__()
        self.feature_dim = feature_dim
        self.input_channels = input_channels

        self.feature_extractor = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=4, dilation=4),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=8, dilation=8),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
        self.reconstructor = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_channels * feature_dim),
        )
        self.projection_head = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
        )

    def forward(self, x, mode="supervised"):
        features = self.feature_extractor(x)
        if mode == "supervised":
            return self.classifier(features)
        if mode == "reconstruction":
            return self.reconstructor(features)
        if mode == "contrastive":
            return self.projection_head(features)
        if mode == "features":
            return features
        raise ValueError("Invalid mode")


class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


class EarlyStopping:
    def __init__(self, patience=5, delta=0.001, mode="max", path="checkpoint.pth"):
        self.patience = patience
        self.delta = delta
        self.mode = mode
        self.path = path
        self.counter = 0
        self.early_stop = False
        self.best_score = -np.inf if mode == "max" else np.inf

    def __call__(self, score, model):
        improved = score > self.best_score + self.delta if self.mode == "max" else score < self.best_score - self.delta
        if improved:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

    def load_best(self, model):
        state = torch.load(self.path, map_location=device)
        model.load_state_dict(state)


# =========================
# Training and evaluation
# =========================
def train_one_epoch_verbose(
    model,
    labeled_loader,
    unlabeled_loader,
    optimizer,
    criterion,
    device,
    lambda_recon=1.0,
    lambda_contrast=0.5,
    temperature=0.1,
):
    model.train()
    labeled_iter = cycle_loader(labeled_loader)
    unlabeled_iter = cycle_loader(unlabeled_loader)
    num_steps = max(len(labeled_loader), len(unlabeled_loader))

    total_loss = 0.0
    total_cls, total_rec, total_con = 0.0, 0.0, 0.0
    all_true, all_pred = [], []

    pbar = tqdm(range(num_steps), desc="Batch progress", ncols=100)
    for _ in pbar:
        x_l, y_l, _ = next(labeled_iter)
        x_u, _, _ = next(unlabeled_iter)

        x_l, y_l, x_u = x_l.to(device), y_l.to(device), x_u.to(device)

        logits = model(x_l, mode="supervised")
        class_loss = criterion(logits, y_l.long())
        pred = torch.argmax(logits, dim=1)
        all_true.extend(y_l.detach().cpu().numpy())
        all_pred.extend(pred.detach().cpu().numpy())

        recon = model(x_u, mode="reconstruction").view_as(x_u)
        recon_loss = F.mse_loss(recon, x_u)

        aug1 = apply_contrastive_augmentation(
            x_u,
            noise_std=CONTRAST_NOISE_STD,
            mask_prob=CONTRAST_MASK_PROB,
            use_noise=USE_CONTRAST_NOISE,
            use_masking=USE_CONTRAST_MASKING,
        )
        aug2 = apply_contrastive_augmentation(
            x_u,
            noise_std=CONTRAST_NOISE_STD,
            mask_prob=CONTRAST_MASK_PROB,
            use_noise=USE_CONTRAST_NOISE,
            use_masking=USE_CONTRAST_MASKING,
        )
        proj1 = F.normalize(model(aug1, mode="contrastive"), dim=1)
        proj2 = F.normalize(model(aug2, mode="contrastive"), dim=1)
        logits_con = torch.mm(proj1, proj2.T) / temperature
        labels_con = torch.arange(proj1.size(0), device=device)
        contrast_loss = 0.5 * (
            F.cross_entropy(logits_con, labels_con) + F.cross_entropy(logits_con.T, labels_con)
        )

        loss = class_loss + lambda_recon * recon_loss + lambda_contrast * contrast_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_cls += class_loss.item()
        total_rec += recon_loss.item()
        total_con += contrast_loss.item()

        pbar.set_postfix(
            {
                "Cls": f"{class_loss.item():.4f}",
                "Rec": f"{recon_loss.item():.4f}",
                "Con": f"{contrast_loss.item():.4f}",
            }
        )

    train_acc = accuracy_score(all_true, all_pred)
    return {
        "loss": total_loss / num_steps,
        "class_loss": total_cls / num_steps,
        "recon_loss": total_rec / num_steps,
        "contrast_loss": total_con / num_steps,
        "train_acc": train_acc,
    }


@torch.no_grad()
def evaluate_metrics_only(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    for x, y, _ in loader:
        mask_cpu = (y != -1).numpy()
        if mask_cpu.sum() == 0:
            continue
        x = x[mask_cpu].to(device)
        y = y[mask_cpu].to(device)

        logits = model(x, mode="supervised")
        probs = F.softmax(logits, dim=1)[:, 1]
        pred = torch.argmax(logits, dim=1)

        y_true.extend(y.detach().cpu().numpy())
        y_pred.extend(pred.detach().cpu().numpy())
        y_prob.extend(probs.detach().cpu().numpy())

    if len(np.unique(y_true)) < 2:
        return {
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "auc": np.nan,
            "ap": np.nan,
        }

    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_prob)
    ap = auc(recall_curve, precision_curve)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "ap": ap,
    }


@torch.no_grad()
def evaluate_and_plot(model, loader, device, plot_curves=False):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    for x, y, _ in loader:
        mask_cpu = (y != -1).numpy()
        if mask_cpu.sum() == 0:
            continue
        x = x[mask_cpu].to(device)
        y = y[mask_cpu].to(device)

        logits = model(x, mode="supervised")
        probs = F.softmax(logits, dim=1)[:, 1]
        pred = torch.argmax(logits, dim=1)

        y_true.extend(y.detach().cpu().numpy())
        y_pred.extend(pred.detach().cpu().numpy())
        y_prob.extend(probs.detach().cpu().numpy())

    if len(np.unique(y_true)) < 2:
        return None

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall_curve, precision_curve)

    if plot_curves:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.plot(fpr, tpr, label=f"ROC AUC={roc_auc:.4f}")
        plt.plot([0, 1], [0, 1], "--", color="gray")
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.title("ROC curve")
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(recall_curve, precision_curve, label=f"PR AUC={pr_auc:.4f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall curve")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }


@torch.no_grad()
def collect_test_predictions(model, loader, metadata_df, device, split_seed):
    """
    Save per-variant predictions for the current test split.
    metadata_df must be aligned with the eval dataset order and reset_index(drop=True).
    """
    model.eval()
    rows = []

    for x, y, sample_idx in loader:
        mask_cpu = (y != -1).numpy()
        if mask_cpu.sum() == 0:
            continue

        kept_idx = sample_idx[mask_cpu].numpy().tolist()
        x = x[mask_cpu].to(device)
        y = y[mask_cpu].to(device)

        logits = model(x, mode="supervised")
        probs = F.softmax(logits, dim=1)[:, 1]
        preds = torch.argmax(logits, dim=1)

        meta_batch = metadata_df.iloc[kept_idx].reset_index(drop=True)

        for i in range(len(kept_idx)):
            row = {
                "split_seed": split_seed,
                "true_label": int(y[i].detach().cpu().item()),
                "pred_label": int(preds[i].detach().cpu().item()),
                "pred_prob": float(probs[i].detach().cpu().item()),
            }
            if not meta_batch.empty:
                row.update(meta_batch.iloc[i].to_dict())
            rows.append(row)

    return pd.DataFrame(rows)


def build_supplementary_prediction_table(all_predictions_df):
    candidate_id_cols = ["variant_index", "Otherinfo", "Phenotype", "ClinVar_label"]
    group_cols = [c for c in candidate_id_cols if c in all_predictions_df.columns]

    if len(group_cols) == 0:
        raise ValueError(
            "No metadata columns available for grouping supplementary predictions. "
            "Expected one of: variant_index, Otherinfo, Phenotype, ClinVar_label."
        )

    summary_pred_df = (
        all_predictions_df.groupby(group_cols, dropna=False)
        .agg(
            true_label=("true_label", "first"),
            mean_pred_prob=("pred_prob", "mean"),
            std_pred_prob=("pred_prob", "std"),
            mean_pred_label=("pred_label", "mean"),
            n_test_appearances=("pred_prob", "count"),
        )
        .reset_index()
        .sort_values(by=["mean_pred_prob"], ascending=False)
    )
    return summary_pred_df


# =========================
# Main experiment
# =========================
os.makedirs("test_set_index", exist_ok=True)
os.makedirs("per_split_predictions", exist_ok=True)

n_splits = 10
split_seeds = range(42, 42 + n_splits)
all_results = []
all_prediction_dfs = []

for split_seed in split_seeds:
    print(f"\n===== Random split seed: {split_seed} =====")
    set_seed(split_seed)

    indices = df.index.values
    labels = df["label"].values
    train_idx, temp_idx, train_y, temp_y = train_test_split(
        indices,
        labels,
        test_size=0.4,
        stratify=labels,
        random_state=split_seed,
    )
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx,
        temp_y,
        test_size=0.5,
        stratify=temp_y,
        random_state=split_seed,
    )
    np.save(f"test_set_index/coding_test_indices_seed{split_seed}.npy", test_idx)

    train_feats = features_all[train_idx]
    train_labels = labels[train_idx]
    val_feats = features_all[val_idx]
    val_labels = labels[val_idx]
    test_feats = features_all[test_idx]
    test_labels = labels[test_idx]

    train_non_numeric = non_numeric_all.iloc[train_idx].copy().reset_index(drop=True)
    val_non_numeric = non_numeric_all.iloc[val_idx].copy().reset_index(drop=True)
    test_non_numeric = non_numeric_all.iloc[test_idx].copy().reset_index(drop=True)

    train_pack = oversample_labeled_subset(train_feats, train_labels, train_non_numeric, seed=split_seed)

    labeled_loader, unlabeled_loader = build_train_loaders(
        train_pack,
        LABELED_BATCH_SIZE,
        UNLABELED_BATCH_SIZE,
        seed=split_seed,
    )
    val_loader = build_eval_loader(val_feats, val_labels, batch_size=BATCH_SIZE)
    test_loader = build_eval_loader(test_feats, test_labels, batch_size=BATCH_SIZE)

    model = SemiSupervisedHybridCNNIDCNN(
        input_channels=1,
        num_classes=2,
        feature_dim=train_feats.shape[1],
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = FocalLoss(alpha=1.0, gamma=2.0)

    checkpoint_path = f"best_model_seed{split_seed}.pth"
    early_stopper = EarlyStopping(
        patience=PATIENCE,
        delta=MIN_DELTA,
        mode="max",
        path=checkpoint_path,
    )

    for epoch in range(1, EPOCHS + 1):
        logs = train_one_epoch_verbose(
            model,
            labeled_loader,
            unlabeled_loader,
            optimizer,
            criterion,
            device,
            lambda_recon=LAMBDA_RECON,
            lambda_contrast=LAMBDA_CONTRAST,
        )
        val_metrics = evaluate_metrics_only(model, val_loader, device)

        print(
            f"Epoch: {epoch} | Loss: {logs['loss']:.4f} | "
            f"Cls: {logs['class_loss']:.4f} | Rec: {logs['recon_loss']:.4f} | "
            f"Con: {logs['contrast_loss']:.4f} | Train Acc: {logs['train_acc'] * 100:.2f}%"
        )
        print(
            f"Val Metrics -> AUC: {val_metrics['auc']:.4f} | AP: {val_metrics['ap']:.4f} | "
            f"Acc: {val_metrics['accuracy']:.4f} | F1: {val_metrics['f1']:.4f}"
        )

        early_stopper(val_metrics["auc"], model)
        if early_stopper.early_stop:
            print(f"Early stopping triggered at epoch {epoch}")
            break

    early_stopper.load_best(model)

    print("\n--- Best Model Test Evaluation ---")
    test_metrics = evaluate_and_plot(model, test_loader, device, plot_curves=PLOT_CURVES)
    test_metrics["split_seed"] = split_seed
    all_results.append(test_metrics)
    print("Test Metrics:", {k: round(v, 4) for k, v in test_metrics.items() if k != "split_seed"})

    test_pred_df = collect_test_predictions(
        model=model,
        loader=test_loader,
        metadata_df=test_non_numeric,
        device=device,
        split_seed=split_seed,
    )
    per_split_path = f"per_split_predictions/test_predictions_seed{split_seed}.csv"
    test_pred_df.to_csv(per_split_path, index=False)
    all_prediction_dfs.append(test_pred_df)
    print(f"Saved per-split predictions to: {per_split_path}")

results_df = pd.DataFrame(all_results)
results_df.to_csv("results_10_splits_noise_masking.csv", index=False)
print("\n===== 10-Fold Summary =====")
print(results_df.describe().loc[["mean", "std"]])

all_predictions_df = pd.concat(all_prediction_dfs, ignore_index=True)
all_predictions_df.to_csv("all_test_predictions_10_splits.csv", index=False)
print("Saved merged prediction file: all_test_predictions_10_splits.csv")

supplementary_pred_df = build_supplementary_prediction_table(all_predictions_df)
supplementary_pred_df.to_csv("supplementary_variant_predictions.csv", index=False)
print("Saved supplementary prediction file: supplementary_variant_predictions.csv")